# Tutorial 4: Agentic Context Engineering (ACE)

**A zero-weight-update approach to continual learning using LLM reflection loops.**

Every strategy we have seen so far modifies the model in some way — TTT-E2E updates weights,
JitRL biases logits, Doc-to-LoRA injects adapter matrices. ACE takes a radically different approach:
**it never touches the model at all.**

Instead, ACE uses an LLM (via Ollama) to iteratively improve a **Playbook** — a JSON collection
of strategies that guide how the model answers questions. The learning happens in the prompt
engineering, not the parameters.

**In this tutorial you will learn:**
1. The Generate-Reflect-Curate loop that drives ACE
2. How the Playbook stores and evolves strategies
3. How to use ACE for document QA with Ollama
4. When zero-weight approaches outperform weight-based ones

## Concept: The Generate-Reflect-Curate Loop

ACE runs multiple loops over QA pairs to evolve its answering strategy:

```
                    +-------------------+
                    |   Document Text   |
                    |   + QA Pairs      |
                    +--------+----------+
                             |
                    +--------v----------+
              +---->|    1. GENERATE     |  Use current playbook + document
              |     |    (answer QA)     |  context to produce an answer
              |     +--------+----------+
              |              |
              |     +--------v----------+
              |     |    2. REFLECT      |  Compare generated answer to
              |     |   (assess quality) |  expected answer, produce feedback
              |     +--------+----------+
              |              |
              |     +--------v----------+
              |     |    3. CURATE       |  Update playbook: add new strategies,
              |     |  (evolve playbook) |  refine existing ones, prune weak ones
              |     +--------+----------+
              |              |
              +--------------+  (repeat for num_loops iterations)
```

Each loop refines the playbook. After several iterations, the playbook contains
battle-tested strategies for answering questions about the document domain.

## Concept: The Playbook — JSON Strategy Persistence

The Playbook is a list of text rules (strategies) that get prepended to the LLM's system prompt.
It is the **learned knowledge** of the ACE system, stored as plain JSON:

```json
{
  "version": 1,
  "strategies": [
    {"id": "s1a2b3c4", "rule": "When asked about dates, cite the specific month and year", "source": "loop_1"},
    {"id": "s5d6e7f8", "rule": "For questions about people, include their title and affiliation", "source": "loop_2"}
  ],
  "stats": {"total_loops": 3, "documents_processed": 1},
  "max_strategies": 50
}
```

Key properties:
- **Portable**: save to disk, load in a new session — no model checkpoint needed
- **Inspectable**: read the strategies in plain English to understand what was learned
- **Bounded**: `max_strategies` caps the playbook size (oldest strategies get evicted)
- **Composable**: merge playbooks from different document domains

In [ ]:
# Cell 4: Prerequisites — check that Ollama is running with the required model
import urllib.request
import json

OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "qwen3.5:9b"

# Check Ollama is running
try:
    req = urllib.request.Request(f"{OLLAMA_URL}/api/tags")
    with urllib.request.urlopen(req, timeout=5) as resp:
        tags = json.loads(resp.read())
    model_names = [m["name"] for m in tags.get("models", [])]
    print(f"Ollama is running. Available models: {model_names}")

    if any(OLLAMA_MODEL.split(":")[0] in name for name in model_names):
        print(f"Model '{OLLAMA_MODEL}' is available.")
    else:
        print(f"WARNING: Model '{OLLAMA_MODEL}' not found. Run: ollama pull {OLLAMA_MODEL}")
except Exception as e:
    print(f"ERROR: Cannot connect to Ollama at {OLLAMA_URL}")
    print(f"Start Ollama with: ollama serve")
    print(f"Then pull the model: ollama pull {OLLAMA_MODEL}")
    raise RuntimeError(f"Ollama not available: {e}")

In [ ]:
# Cell 5: Create ACEEngine with OllamaClient
from continual_learning.ace.engine import ACEEngine
from continual_learning.ace.ollama_client import OllamaClient

# Test the OllamaClient directly first
client = OllamaClient(base_url=OLLAMA_URL, model=OLLAMA_MODEL)
test_response = client.generate("Say hello in exactly 5 words.", temperature=0.3)
print(f"OllamaClient test: {test_response[:200]}")

# Create the ACE engine
engine = ACEEngine(
    ollama_model=OLLAMA_MODEL,
    ollama_base_url=OLLAMA_URL,
    num_loops=3,             # Number of Generate-Reflect-Curate iterations
    playbook_dir="playbooks", # Where to save playbooks
    max_strategies=50,        # Cap on playbook size
)
print(f"\nACEEngine created:")
print(f"  Model: {OLLAMA_MODEL}")
print(f"  Loops: {engine.num_loops}")
print(f"  Playbook strategies: {len(engine._playbook.strategies)}")

In [ ]:
# Cell 6: Learn a document — watch the Generate-Reflect-Curate loop
from pathlib import Path

document = Path("data/sample_document.txt").read_text()
print(f"Document: {len(document.split())} words")
print(f"Preview: {document[:150]}...\n")

# QA pairs for the ACE loop to train on
qa_pairs = [
    {"question": "Who led the Quantum Computing Research Division?", "answer": "Dr. Sarah Chen"},
    {"question": "What was the code name of the 72-qubit processor?", "answer": "RedShift"},
    {"question": "What year was Oracle's quantum division established?", "answer": "2019"},
]

# Progress callback to see the loop in action
def progress(loop_num, assessment):
    print(f"  Loop {loop_num}: {assessment[:100]}")

print("Starting ACE learning (Generate-Reflect-Curate loops)...")
print("-" * 60)
result = engine.learn(document, qa_pairs=qa_pairs, callback=progress)
print("-" * 60)

print(f"\nResult:")
print(f"  Method: {result['method']}")
print(f"  Loops completed: {result['loops_completed']}")
print(f"  Strategies learned: {result['num_strategies']}")
print(f"  Tokens processed: {result['tokens_processed']}")

In [ ]:
# Cell 7: Query about the learned content
test_questions = [
    "Who led the quantum computing research at Oracle Labs?",
    "What was the RedShift processor's quantum volume?",
    "What post-quantum encryption was integrated into OCI?",
    "How many researchers were in the division by end of 2024?",
]

print("Querying with evolved playbook")
print("=" * 60)
for q in test_questions:
    answer = engine.generate(q)
    print(f"\nQ: {q}")
    print(f"A: {answer[:300]}")

In [ ]:
# Cell 8: Save/load the playbook and inspect its strategies
import os

# Save the playbook
os.makedirs("playbooks", exist_ok=True)
save_path = engine.save_playbook("quantum_computing")
print(f"Playbook saved to: {save_path}")

# Inspect the playbook contents
playbook = engine._playbook
print(f"\nPlaybook stats:")
print(f"  Created: {playbook.created}")
print(f"  Updated: {playbook.updated}")
print(f"  Documents processed: {playbook.stats['documents_processed']}")
print(f"  Total strategies: {len(playbook.strategies)}")

print(f"\nStrategies:")
for i, s in enumerate(playbook.strategies):
    print(f"  [{s['id']}] (source: {s['source']})")
    print(f"    {s['rule'][:120]}")

print(f"\nRendered playbook (what the LLM sees):")
print("-" * 40)
print(playbook.render())

# Demonstrate loading into a fresh engine
engine2 = ACEEngine(ollama_model=OLLAMA_MODEL, ollama_base_url=OLLAMA_URL)
engine2.load_playbook("quantum_computing")
print(f"\nLoaded playbook into fresh engine: {len(engine2._playbook.strategies)} strategies")

In [ ]:
# Cell 9: Multi-document learning — watch the playbook evolve
# Create a second document with different content
document_2 = """Oracle Cloud Infrastructure (OCI) Networking Overview

OCI's Virtual Cloud Network (VCN) provides a customizable, private network in Oracle Cloud.
Each VCN has a CIDR block (e.g., 10.0.0.0/16) and can contain multiple subnets across
availability domains. Security lists and network security groups control ingress and egress
traffic at the packet level.

FastConnect provides a dedicated, private connection between an on-premises data center and OCI,
bypassing the public internet. It supports bandwidth from 1 Gbps to 100 Gbps and offers
lower latency and more consistent networking compared to IPSec VPN tunnels.

The OCI Load Balancer distributes traffic across multiple compute instances. It supports
both Layer 4 (TCP) and Layer 7 (HTTP) load balancing, with health checks, SSL termination,
and session persistence. The flexible shape allows scaling from 10 Mbps to 8 Gbps."""

qa_pairs_2 = [
    {"question": "What is a VCN in OCI?", "answer": "Virtual Cloud Network"},
    {"question": "What bandwidth does FastConnect support?", "answer": "1 Gbps to 100 Gbps"},
]

print(f"Before: {len(engine._playbook.strategies)} strategies")

print("\nLearning document 2...")
result2 = engine.learn(document_2, qa_pairs=qa_pairs_2, callback=progress)

print(f"\nAfter: {len(engine._playbook.strategies)} strategies")
print(f"Documents in engine: {engine.num_documents}")
print(f"Loops completed: {result2['loops_completed']}")

# Test cross-document queries
print("\nCross-document queries:")
for q in ["What is FastConnect?", "Who led the quantum division?"]:
    answer = engine.generate(q)
    print(f"  Q: {q}")
    print(f"  A: {answer[:200]}\n")

## When to Use ACE

**ACE is ideal when:**
- You cannot modify the model (API-only access, or compliance constraints)
- You want human-inspectable learned knowledge (the playbook is plain text)
- You need portable knowledge (JSON file, no GPU-dependent checkpoints)
- The task benefits from meta-learning (learning *how* to answer, not *what* to answer)

**ACE is less suitable when:**
- You need sub-second latency (each loop requires multiple LLM calls)
- You do not have access to an LLM inference server (Ollama, vLLM, etc.)
- The task requires memorizing exact facts (ACE learns *strategies*, not *facts*)
- You need offline/embedded deployment without network access

**Comparison with other strategies:**

| Strategy | Modifies Model? | Requires GPU? | Knowledge Format | Learning Speed |
|----------|----------------|---------------|-----------------|---------------|
| TTT-E2E | Yes (weights) | Yes | Model checkpoint | Medium |
| JitRL MVP | No (retrieval) | For generation | TF-IDF index | Instant |
| JitRL Full | No (logit mod) | Yes | Hidden-state embeddings | Fast |
| Doc-to-LoRA | Yes (adapter) | Yes | LoRA matrices | Fast |
| **ACE** | **No** | **No (Ollama)** | **JSON playbook** | **Slow (loops)** |

## Exercises

1. **Loop count experiment**: Try `num_loops=1`, `num_loops=5`, and `num_loops=10`.
   How does the number of loops affect the quality and diversity of playbook strategies?
   Is there diminishing returns after a certain point?

2. **Playbook transfer**: Learn a playbook on document A, save it, then load it when
   querying about a completely different document B. Does the playbook's meta-knowledge
   (answering strategies) transfer across domains?

3. **Strategy analysis**: After learning, manually read each strategy in the playbook.
   Can you identify which strategies are genuinely useful vs. which are noise?
   Try manually removing weak strategies and re-testing.

4. **Model comparison**: Try ACE with different Ollama models (e.g., `qwen3.5:4b` vs. `qwen3.5:9b`).
   Does a larger reflection model produce better strategies?

5. **Hybrid approach**: Use ACE to learn strategies, then use those strategies as a system
   prompt for JitRL MVP's context window. Does combining approaches help?

In [ ]:
# Cell 12: Clean up
engine.clear()
print(f"Engine cleared. Documents: {engine.num_documents}, Strategies: {len(engine._playbook.strategies)}")

# Clean up saved playbook
import shutil
if os.path.exists("playbooks"):
    shutil.rmtree("playbooks")
    print("Cleaned up playbooks directory.")